## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутстввовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

Постарайтесь провалидировать код с помощью assert-ов, вызываемых в ноутбуке или в .py-файлах.

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

In [1]:
from json_dict_processing import create_processor

test_data = {'field_1': {'f2' : 2}, 'test' : '22'}
CONFIG_TEST = {
    'quantity': ['field_1', 'f2'],
    'price': (['test'], lambda x: int(x))
    }

def test_processor():
    return create_processor(CONFIG_TEST, list_processor=False)

assert test_processor()(test_data) == {'quantity': 2, 'price': 22}  # сработало! 

In [2]:
from laureates_data_loader import get_list_of_laureate_data
laureates = get_list_of_laureate_data()



### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

1. Всего записей:

In [3]:
len(laureates)

1018

2-3. Все ли значения полей заполнены?

In [4]:
from dataset_helpers import select_field, filter_dicts

persons = filter_dicts(laureates, {'type_': 'person'})
organizations = filter_dicts(laureates, {'type_': 'org'})

In [5]:
print('Статистика для лауреатов-людей:')
for field in persons[0].keys():
    res = select_field(persons, field)
    missing = sum([1 for i in res if i is None])
    print(f'\t{field}: missing {missing} values out of {len(res)}')

print('Статистика для лауреатов-организаций:')
for field in organizations[0].keys():
    res = select_field(organizations, field)
    missing = sum([1 for i in res if i is None])
    print(f'\t{field}: missing {missing} values out of {len(res)}')

Статистика для лауреатов-людей:
	id: missing 0 values out of 990
	name: missing 0 values out of 990
	gender: missing 0 values out of 990
	birth_year: missing 0 values out of 990
	country_birth: missing 3 values out of 990
	country_now: missing 3 values out of 990
	prizes_relevant: missing 0 values out of 990
	type_: missing 0 values out of 990
Статистика для лауреатов-организаций:
	id: missing 0 values out of 28
	name: missing 0 values out of 28
	founded_year: missing 1 values out of 28
	country_founded: missing 4 values out of 28
	country_now: missing 4 values out of 28
	prizes_relevant: missing 0 values out of 28
	type_: missing 0 values out of 28


4.1. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? 

In [6]:
from aggregations import top_n
biggest_5 = top_n(laureates, 'id', 5)
smallest_5 = top_n(laureates, 'id', 5, reverse=False)

years_big_5 = []
for awardee in biggest_5:  # возможно стоит добавить отдельную функцию для обработки свойств наград
    for prize in awardee['prizes_relevant']:
        years_big_5.append(prize['award_year'])

years_sm_5 = []
for awardee in smallest_5: 
    for prize in awardee['prizes_relevant']:
        years_sm_5.append(prize['award_year'])

print(f'Года для 5 наибольших id: {years_big_5}, значения id: {select_field(biggest_5, 'id')}')
print(f'Года для 5 наименьших id: {years_sm_5}, значения id: {select_field(smallest_5, 'id')}')

Года для 5 наибольших id: [2025, 2025, 2025, 2025, 2025], значения id: [1060, 1059, 1058, 1057, 1056]
Года для 5 наименьших id: [1901, 1902, 1902, 1903, 1903], значения id: [1, 2, 3, 4, 5]


Похоже, что id присваивают по году вручения награды

4.2. Есть ли записи с повторяющимися id?

In [7]:
ids = select_field(laureates, 'id')
len(ids) == len(set(ids))

True

Все ID уникальны, что в целом логично.

4.3. Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

In [8]:
s_ids = sorted(ids)
for i in range(1, len(s_ids)):
    if s_ids[i] - s_ids[i - 1] > 1:
        print(f'Да, есть, например: {s_ids[i]} и {s_ids[i - 1]}')
        break

Да, есть, например: 8 и 6



Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

In [9]:
from dataset_helpers import top_n_freq
top_countries = top_n_freq(laureates, 'country_now', 5)
print('Топ-5 стран по количеству лауреатов (не призов, а лауреатов) (будем считать на момент вручения)')
for i, (country, count) in enumerate(top_countries, 1):
    print(f"{i}. {country}: {count} лауреатов")

Топ-5 стран по количеству лауреатов (не призов, а лауреатов) (будем считать на момент вручения)
1. USA: 303 лауреатов
2. United Kingdom: 96 лауреатов
3. Germany: 84 лауреатов
4. France: 65 лауреатов
5. Japan: 31 лауреатов


Теперь посмотрим топ 1 по призам (!) по категориям

In [10]:
from dataset_helpers import flatten_prizes, groupby

prizes = flatten_prizes(laureates)
prizes_by_category = groupby(prizes, ['category_en'])
for category, category_prizes in prizes_by_category.items():
    print(f"{category[0]}:")
    country_stats = top_n_freq(category_prizes, 'country_now', 5)
    for i, (country, count) in enumerate(country_stats, 1):
        print(f"\t{i}. {country}: {count}")

Economic Sciences:
	1. USA: 54
	2. United Kingdom: 7
	3. Canada: 5
	4. France: 5
	5. the Netherlands: 4
Physics:
	1. USA: 72
	2. Germany: 27
	3. United Kingdom: 25
	4. France: 13
	5. Japan: 12
Chemistry:
	1. USA: 60
	2. United Kingdom: 27
	3. Germany: 26
	4. France: 11
	5. Japan: 8
Literature:
	1. France: 12
	2. USA: 10
	3. Sweden: 7
	4. Poland: 7
	5. Germany: 7
Peace:
	1. USA: 26
	2. France: 12
	3. Switzerland: 11
	4. United Kingdom: 7
	5. Russia: 5
Physiology or Medicine:
	1. USA: 84
	2. United Kingdom: 25
	3. Germany: 18
	4. France: 12
	5. Sweden: 8


**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

In [11]:
from attribute_manager import add_field
from aggregations import agg_mean, agg_median

def get_first_prize_age(person):
    birth_year = person['birth_year']
    prizes = person['prizes_relevant']
    if birth_year and prizes:
        first_prize_year = min(prize['award_year'] for prize in prizes)
        return first_prize_year - birth_year
    return None

persons_aged = add_field(persons, 'first_prize_age', get_first_prize_age)

In [12]:
print(f'Средний возраст на момент получения 1ой премии: {agg_mean(persons_aged, 'first_prize_age'):.1f} лет')

Средний возраст на момент получения 1ой премии: 60.4 лет


In [13]:
from dataset_helpers import groupby

for person in persons_aged:
    if person['prizes_relevant']:
        first_prize = min(person['prizes_relevant'], key=lambda x: x.get('award_year', 9999))
        person['first_prize_category'] = first_prize.get('category_en')

age_prize_category = groupby(persons_aged, ['first_prize_category'])
for category, group in age_prize_category.items():
    cat_name = category[0]
    mean_age = agg_mean(group, 'first_prize_age')
    median_age = agg_median(group, 'first_prize_age')
    print(f"{cat_name}: средний {mean_age:.1f} лет, медиана {median_age:.1f} лет")

Economic Sciences: средний 67.0 лет, медиана 67.0 лет
Physics: средний 57.7 лет, медиана 56.0 лет
Chemistry: средний 59.2 лет, медиана 58.0 лет
Literature: средний 65.0 лет, медиана 67.0 лет
Peace: средний 60.8 лет, медиана 62.0 лет
Physiology or Medicine: средний 58.9 лет, медиана 58.0 лет


Средний и медианный совпадают только у экономистов, у других больше/меньше, но всегда в пределах +- 2 лет. 

2.3. Гендерное распределение

Как менялось количество женщин-лауреатов по десятилетиям?
Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

In [14]:
from dataset_helpers import flatten_prizes, groupby, filter_dicts
from attribute_manager import add_field

prizes = flatten_prizes(laureates)
prizes = add_field(prizes, 'decade', lambda x : x['award_year'] // 10 if x['award_year'] else None)

person_prizes = filter_dicts(prizes, {'laureate_type' : 'person'})
women_prizes = filter_dicts(person_prizes, {'gender' : 'female'})

def show_by_year(input_data):
    age_prize_category = groupby(input_data, ['decade'])
    for dec, group in sorted(age_prize_category.items(), key = lambda x: -x[0][0]):
        print(f"{dec[0]*10}-{dec[0]*10+9}: {len(group)}")

print('Количество женщин-лауреаток Нобелевской премии по десятилетиям:')
show_by_year(women_prizes)

Количество женщин-лауреаток Нобелевской премии по десятилетиям:
2020-2029: 14
2010-2019: 13
2000-2009: 11
1990-1999: 7
1980-1989: 4
1970-1979: 4
1960-1969: 3
1940-1949: 3
1930-1939: 3
1920-1929: 2
1910-1919: 1
1900-1909: 3


Как мы видим, в среднем количество женщин-лауреаток растет.

In [15]:
def show_by_category(input_data):
    category_prize_category = groupby(input_data, ['category_en'])
    for cat, group in category_prize_category.items():
        print(f"{cat[0]}: {len(group)}")
        
print('Количество женщин-лауреаток Нобелевской премии по категориям:')
show_by_category(women_prizes)

Количество женщин-лауреаток Нобелевской премии по категориям:
Chemistry: 8
Literature: 18
Peace: 20
Physics: 5
Physiology or Medicine: 14
Economic Sciences: 3


Женщины значительно чаще получают нобелевскую премию Мира, по Литературе и Психологии, по сравнению с другими категориями.

In [16]:
print('Количество женщин-лауреаток Нобелевской премии по десятилетиям и категориям:')
age_prize_category = groupby(women_prizes, ['decade'])
for dec, group in sorted(age_prize_category.items(), key = lambda x: -x[0][0]):
    print(f"{dec[0]*10}-{dec[0]*10+9}:")
    category_prize_category = groupby(group, ['category_en'])
    for cat, group in category_prize_category.items():
        print(f"\t{cat[0]}: {len(group)}")

Количество женщин-лауреаток Нобелевской премии по десятилетиям и категориям:
2020-2029:
	Physics: 2
	Literature: 3
	Chemistry: 3
	Economic Sciences: 1
	Physiology or Medicine: 2
	Peace: 3
2010-2019:
	Literature: 3
	Physics: 1
	Peace: 5
	Economic Sciences: 1
	Chemistry: 1
	Physiology or Medicine: 2
2000-2009:
	Chemistry: 1
	Physiology or Medicine: 4
	Literature: 3
	Economic Sciences: 1
	Peace: 2
1990-1999:
	Peace: 3
	Physiology or Medicine: 1
	Literature: 3
1980-1989:
	Peace: 1
	Physiology or Medicine: 3
1970-1979:
	Peace: 3
	Physiology or Medicine: 1
1960-1969:
	Chemistry: 1
	Physics: 1
	Literature: 1
1940-1949:
	Peace: 1
	Literature: 1
	Physiology or Medicine: 1
1930-1939:
	Chemistry: 1
	Peace: 1
	Literature: 1
1920-1929:
	Literature: 2
1910-1919:
	Chemistry: 1
1900-1909:
	Peace: 1
	Physics: 1
	Literature: 1


Видно, что в последние десятилетия разнообразие категорий сильно больше, но в среднем, те же категории что и раньше остаются самыми популярными.

---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций**

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-IDF + сравнените топ слов по этой метрике с просто самыми популярными словами)